# Marketing Campagin Analysis

## This dataset is about Customer Response 
#### we find who is the customers , what they actualy buy ? how they buy ? and whether they responded to each marketing campgain or not 

In [1]:
#Importing libraries
import pandas as pd
import numpy as np


In [2]:
# Step 1: Load (already done, but shown for clarity)
data = pd.read_csv("../data/marketing_campaign.csv")

# Step 2: Extract correct column names from the single column header
cols = data.columns[0].split(';')

# Step 3: Split the data into multiple columns
data = data.iloc[:, 0].str.split(';', expand=True)

# Step 4: Assign correct column names
data.columns = cols

# Step 5: Reset index (optional but clean)
data = data.reset_index(drop=True)



the file uses semicolon(;) insead of comma , so we basically used split(;) 

In [3]:
data.shape

(2240, 29)

In [4]:
data.columns

Index(['ID', 'Year_Birth', 'Education', 'Marital_Status', 'Income', 'Kidhome',
       'Teenhome', 'Dt_Customer', 'Recency', 'MntWines', 'MntFruits',
       'MntMeatProducts', 'MntFishProducts', 'MntSweetProducts',
       'MntGoldProds', 'NumDealsPurchases', 'NumWebPurchases',
       'NumCatalogPurchases', 'NumStorePurchases', 'NumWebVisitsMonth',
       'AcceptedCmp3', 'AcceptedCmp4', 'AcceptedCmp5', 'AcceptedCmp1',
       'AcceptedCmp2', 'Complain', 'Z_CostContact', 'Z_Revenue', 'Response'],
      dtype='object')

1. ID, Year_birth , Education , Marital_status , Income are all demographic("ko ho ta customer ?" )

( Older, higher-income, and more educated customers may respond differently to campaigns. This is our demographic segmentation layer.)

2. 'Kidhome', 'Teenhome' ("What does their home look like ?")

Customers with kids are often budget-conscious and may respond less to premium offers. A household with no kids and high income is a prime target.

3. Dt_customer , Recency ("Customer Ko Relationship")

 A customer who bought 5 days ago is very different from one who bought 300 days ago. Recency is one of the most powerful indicators of engagement. We'll later engineer an Age column from Year_Birth and a Days_as_Customer from Dt_Customer.

4. 'MntWines', 'MntFruits','MntMeatProducts', 'MntFishProducts', 'MntSweetProducts','MntGoldProds''

Total spending = MntWines + MntFruits + MntMeat + MntFish + MntSweets + MntGold. This becomes our "Total Spend" feature a key indicator of customer value.

5. 'NumDealsPurchases', 'NumWebPurchases','NumCatalogPurchases', 'NumStorePurchases', 'NumWebVisitsMonth'(How did they purchase ?)

 A customer who only buys on deals is very different from one who buys at full price. Channel preference also tells us where to target them.

6. 'AcceptedCmp3', 'AcceptedCmp4', 'AcceptedCmp5', 'AcceptedCmp1', 'AcceptedCmp2', 'Complain', 'Z_CostContact', 'Z_Revenue', 'Response'(Did they response to campagins ?? target variables haru )

Response is our main target column. Everything else helps us predict and explain why a customer said yes or no to the last campaign.

In [9]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2240 entries, 0 to 2239
Data columns (total 29 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   ID                   2240 non-null   object
 1   Year_Birth           2240 non-null   object
 2   Education            2240 non-null   object
 3   Marital_Status       2240 non-null   object
 4   Income               2240 non-null   object
 5   Kidhome              2240 non-null   object
 6   Teenhome             2240 non-null   object
 7   Dt_Customer          2240 non-null   object
 8   Recency              2240 non-null   object
 9   MntWines             2240 non-null   object
 10  MntFruits            2240 non-null   object
 11  MntMeatProducts      2240 non-null   object
 12  MntFishProducts      2240 non-null   object
 13  MntSweetProducts     2240 non-null   object
 14  MntGoldProds         2240 non-null   object
 15  NumDealsPurchases    2240 non-null   object
 16  NumWeb

we see all columns are object which is tottaly wrong . we need to the data type of each column according to their data type .

In [6]:
data.describe().round(2)

,ID,Year_Birth,Education,Marital_Status,Income,Kidhome,Teenhome,Dt_Customer,Recency,MntWines,...,NumWebVisitsMonth,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Complain,Z_CostContact,Z_Revenue,Response
count,2240,2240,2240,2240,2240,2240,2240,2240,2240,2240,...,2240,2240,2240,2240,2240,2240,2240,2240,2240,2240
unique,2240,59,5,8,1975,3,3,663,100,776,...,16,2,2,2,2,2,2,1,1,2
top,1448,1976,Graduation,Married,,0,0,2012-08-31,56,2,...,7,0,0,0,0,0,0,3,11,0
freq,1,89,1127,864,24,1293,1158,12,37,42,...,393,2077,2073,2077,2096,2210,2219,2240,2240,1906


In [7]:
# Checking the missing values 

missing_values = data.isna().sum()
print("There are :\n",missing_values)

There are :
 ID                     0
Year_Birth             0
Education              0
Marital_Status         0
Income                 0
Kidhome                0
Teenhome               0
Dt_Customer            0
Recency                0
MntWines               0
MntFruits              0
MntMeatProducts        0
MntFishProducts        0
MntSweetProducts       0
MntGoldProds           0
NumDealsPurchases      0
NumWebPurchases        0
NumCatalogPurchases    0
NumStorePurchases      0
NumWebVisitsMonth      0
AcceptedCmp3           0
AcceptedCmp4           0
AcceptedCmp5           0
AcceptedCmp1           0
AcceptedCmp2           0
Complain               0
Z_CostContact          0
Z_Revenue              0
Response               0
dtype: int64


data set seem to be clear as there is no any missing values in each coulumns 

### Unique values in categorical coulumns

In [8]:
#Checking the missing values in categorical columns

catg_cols = ['Education','Marital_Status']

for col in catg_cols:
     print(f"\n{col}-{data[col].nunique()}")
     print(data[col].value_counts())


Education-5
Education
Graduation    1127
PhD            486
Master         370
2n Cycle       203
Basic           54
Name: count, dtype: int64

Marital_Status-8
Marital_Status
Married     864
Together    580
Single      480
Divorced    232
Widow        77
Alone         3
Absurd        2
YOLO          2
Name: count, dtype: int64


There is 5 unique values in education and 8 unique value in marital status where Graudation rate is higher than other education .also, There is large number of people who are married (864) and together (580)

## key findings

- dataset size 2240 rows and 29 columns
- missing values 0